# Data Preparation

## 1. Load Weekly Route Operations Data
Load `weekly_route_operations.csv`

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

weekly_route_operations_df = pd.read_csv("../data/weekly_route_operations.csv")
weekly_route_operations_df.head()

,date,route_id,trade_volume_tonnes,shipping_delay_days,freight_cost_usd,container_availability_index,port_congestion_index,fuel_cost_index,commodity_price_index,weather_disruption_score,geopolitical_risk_score,route_status,carbon_emissions_tonnes
0,1/4/2015,R00001,8418.69,7.55,4586.66,71.42,82.22,62.48,19.20,59.51,57.15,Delayed,3681.49
1,1/11/2015,R00001,9343.00,9.33,4574.70,79.27,72.01,59.31,35.05,91.27,41.47,Delayed,4085.69
2,1/18/2015,R00001,7090.69,3.67,4520.46,60.16,44.64,63.24,74.22,46.69,79.69,Normal,3100.76
3,1/25/2015,R00001,7829.79,9.22,4696.95,66.43,97.82,67.62,68.59,56.39,67.15,Delayed,3423.97
4,2/1/2015,R00001,11339.59,6.36,4507.98,96.20,73.21,58.83,28.98,95.92,7.59,Delayed,4958.80


## 2. Fixing Dates & Perform Sorting
`date` loads as text, which won't sort chronologically and won't match the `date` columns in the other files when merging. Convert it to a real date type with `pd.to_datetime`.

In [27]:
weekly_route_operations_df["date"] = pd.to_datetime(weekly_route_operations_df["date"], format="%m/%d/%Y")
weekly_route_operations_df = weekly_route_operations_df.sort_values(["route_id", "date"]).reset_index(drop=True)

print("Date range:", weekly_route_operations_df["date"].min().date(), "to", weekly_route_operations_df["date"].max().date())

Date range: 2015-01-04 to 2026-12-27


## 3. Check Shape & Column Types
We expect there to be 31,300 rows and 13 columns

In [28]:
print("Shape:", weekly_route_operations_df.shape)
weekly_route_operations_df.info()

Shape: (31300, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31300 entries, 0 to 31299
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   date                          31300 non-null  datetime64[ns]
 1   route_id                      31300 non-null  object        
 2   trade_volume_tonnes           31300 non-null  float64       
 3   shipping_delay_days           31300 non-null  float64       
 4   freight_cost_usd              31300 non-null  float64       
 5   container_availability_index  31300 non-null  float64       
 6   port_congestion_index         31300 non-null  float64       
 7   fuel_cost_index               31300 non-null  float64       
 8   commodity_price_index         31300 non-null  float64       
 9   weather_disruption_score      31300 non-null  float64       
 10  geopolitical_risk_score       31300 non-null  float64       
 11  route_sta

## 4. Checking Missing Values & Duplicates
Each `(route_id, date)` pair should appear exactly once: 50 routes x 626 weeks.

In [29]:
print("Missing values:", weekly_route_operations_df.isna().sum().sum())
print("Duplicate rows:", weekly_route_operations_df.duplicated().sum())
print("Duplicate route-weeks:", weekly_route_operations_df.duplicated(["route_id", "date"]).sum())
print("Routes:", weekly_route_operations_df["route_id"].nunique(), "| Weeks:", weekly_route_operations_df["date"].nunique())

Missing values: 0
Duplicate rows: 0
Duplicate route-weeks: 0
Routes: 50 | Weeks: 626


## 5. Target Distribution
How often is each `route_status` seen?

In [30]:
route_status_counts = weekly_route_operations_df["route_status"].value_counts()
route_status_percent = (route_status_counts / len(weekly_route_operations_df) * 100).round(1)

pd.DataFrame({"count": route_status_counts, "percent": route_status_percent}).sort_values("count", ascending=False)

,count,percent
route_status,,
Delayed,20038,64.0
Normal,10264,32.8
Disrupted,998,3.2


## 6. Load Other Files
Load the 4 supporting files: `trade_routes.csv`, `country_metadata.csv`, `commodity_market.csv` and `geopolitical_events.csv`.

Side Note: `commodity_market.csv` is read with `encoding="latin-1"` because its first header contains a byte that isn't valid UTF-8.

In [31]:
trade_routes_df = pd.read_csv("../data/trade_routes.csv")
country_metadata_df = pd.read_csv("../data/country_metadata.csv")
commodity_market_df = pd.read_csv("../data/commodity_market.csv", encoding="latin-1")
geopolitical_events_df = pd.read_csv("../data/geopolitical_events.csv")

print("commodity_market columns:", list(commodity_market_df.columns))

commodity_market columns: ['REU\xa0', 'oil_price', 'natural_gas_price', 'steel_price', 'wheat_price', 'copper_price', 'commodity_stress_index']


### Fix Commodity Header & Dates
The first column of `commodity_market.csv` is named `REU\xa0` instead of `date`. We need to rename it and convert both date columns.
Convert date style for `geopolitical_events.csv`

In [32]:
commodity_market_df = commodity_market_df.rename(columns={commodity_market_df.columns[0]: "date"})
commodity_market_df["date"] = pd.to_datetime(commodity_market_df["date"], format="%m/%d/%Y")
geopolitical_events_df["date"] = pd.to_datetime(geopolitical_events_df["date"], format="ISO8601")

### Profile Each File
Run the same checks as sections 3 and 4, but on all four supporting files at once: rows, columns, total missing values and duplicate rows, gathered into one table. The printed date ranges confirm `commodity_market` and `geopolitical_events` cover the weeks in the operations data, so the merges ahead don't leave gaps.

In [33]:
other_dfs = {
    "trade_routes": trade_routes_df,
    "country_metadata": country_metadata_df,
    "commodity_market": commodity_market_df,
    "geopolitical_events": geopolitical_events_df,
}

profile = pd.DataFrame({
    "rows": {name: df.shape[0] for name, df in other_dfs.items()},
    "columns": {name: df.shape[1] for name, df in other_dfs.items()},
    "missing_values": {name: df.isna().sum().sum() for name, df in other_dfs.items()},
    "duplicate_rows": {name: df.duplicated().sum() for name, df in other_dfs.items()},
})

print("commodity_market dates:", commodity_market_df["date"].min().date(), "to", commodity_market_df["date"].max().date())
print("geopolitical_events dates:", geopolitical_events_df["date"].min().date(), "to", geopolitical_events_df["date"].max().date())
profile

commodity_market dates: 2015-01-04 to 2026-12-27
geopolitical_events dates: 2015-01-01 to 2026-12-31


,rows,columns,missing_values,duplicate_rows
trade_routes,50,7,0,0
country_metadata,10,9,0,0
commodity_market,626,7,0,0
geopolitical_events,10003,7,0,0


## 7. Check Join Keys
Before merging, confirm that:
- each lookup table has one row per key, so a merge can't duplicate rows
- every key in `weekly_route_operations_df` exists in the table it joins to

In [34]:
print("route_id unique in trade_routes:", trade_routes_df["route_id"].is_unique)    # key is set as the route_id
print("country unique in country_metadata:", country_metadata_df["country"].is_unique)  # key is set as the country
print("date unique in commodity_market:", commodity_market_df["date"].is_unique)    # key is set as the date

route_countries = set(trade_routes_df["origin_country"]) | set(trade_routes_df["destination_country"])
print("Routes missing from trade_routes:", set(weekly_route_operations_df["route_id"]) - set(trade_routes_df["route_id"]))
print("Countries missing from country_metadata:", route_countries - set(country_metadata_df["country"]))
print("Weeks missing from commodity_market:", len(set(weekly_route_operations_df["date"]) - set(commodity_market_df["date"])))
print("Duplicate origin -> destination pairs:", trade_routes_df.duplicated(["origin_country", "destination_country"]).sum())

route_id unique in trade_routes: True
country unique in country_metadata: True
date unique in commodity_market: True
Routes missing from trade_routes: set()
Countries missing from country_metadata: set()
Weeks missing from commodity_market: 0
Duplicate origin -> destination pairs: 8


## 8. Join Route and Country Details
Merging columns from `trade_routes.csv` to respective rows located in `weekly_route_operations.csv` based on route_id

In [35]:
merged_df = (
    weekly_route_operations_df
    .merge(trade_routes_df, on="route_id", how="left", validate="many_to_one")
    .merge(country_metadata_df.add_prefix("origin_"), on="origin_country", how="left", validate="many_to_one")
    .merge(country_metadata_df.add_prefix("destination_"), on="destination_country", how="left", validate="many_to_one")
)

print("Shape:", merged_df.shape)
merged_df[["date", "route_id", "origin_country", "origin_region", "destination_country", "destination_region", "shipping_method"]].head()

Shape: (31300, 35)


,date,route_id,origin_country,origin_region,destination_country,destination_region,shipping_method
0,2015-01-04,R00001,India,South Asia,United States,North America,Rail
1,2015-01-11,R00001,India,South Asia,United States,North America,Rail
2,2015-01-18,R00001,India,South Asia,United States,North America,Rail
3,2015-01-25,R00001,India,South Asia,United States,North America,Rail
4,2015-02-01,R00001,India,South Asia,United States,North America,Rail


## 9. Join Commodity Prices
Note: `weekly_route_operations.csv` already has `fuel_cost_index` and `commodity_price_index`.
Compare values of `oil_price` from `weekly_route_operations.csv` with `commodity_stress_index` from `commodity_market.csv`.

In [36]:
commodity_check_df = merged_df[["date", "fuel_cost_index", "commodity_price_index"]].drop_duplicates("date").merge(commodity_market_df, on="date")
commodity_check_df["oil_matches"] = commodity_check_df["fuel_cost_index"].round(2) == commodity_check_df["oil_price"].round(2)
commodity_check_df["stress_matches"] = commodity_check_df["commodity_price_index"].round(2) == commodity_check_df["commodity_stress_index"].round(2)

week_count = len(commodity_check_df)
for label, match_column in [("fuel_cost_index == oil_price", "oil_matches"),
                            ("commodity_price_index == commodity_stress_index", "stress_matches")]:
    matching_weeks = commodity_check_df[match_column].sum()
    print(f"{label}: {matching_weeks} of {week_count} weeks ({matching_weeks / week_count:.1%})")

# List every week where either pair disagrees, so the mismatch can be inspected directly.
mismatch_columns = ["date", "fuel_cost_index", "oil_price", "commodity_price_index", "commodity_stress_index"]
mismatched_weeks = commodity_check_df.loc[~commodity_check_df["oil_matches"] | ~commodity_check_df["stress_matches"], mismatch_columns].set_index("date")
mismatched_weeks["oil_ratio"] = mismatched_weeks["oil_price"] / mismatched_weeks["fuel_cost_index"]
mismatched_weeks["stress_ratio"] = mismatched_weeks["commodity_stress_index"] / mismatched_weeks["commodity_price_index"]

print("Mismatched weeks:", len(mismatched_weeks), "out of", len(commodity_check_df))
mismatched_weeks.round(2)

fuel_cost_index == oil_price: 596 of 626 weeks (95.2%)
commodity_price_index == commodity_stress_index: 596 of 626 weeks (95.2%)
Mismatched weeks: 30 out of 626


,fuel_cost_index,oil_price,commodity_price_index,commodity_stress_index,oil_ratio,stress_ratio
date,,,,,,
2026-06-07,58.65,93.84,51.41,64.26,1.6,1.25
2026-06-14,55.11,88.17,57.41,71.76,1.6,1.25
2026-06-21,57.78,92.45,44.47,55.59,1.6,1.25
2026-06-28,61.89,99.02,71.09,88.86,1.6,1.25
2026-07-05,63.78,102.06,31.37,39.21,1.6,1.25
2026-07-12,55.39,88.62,52.52,65.65,1.6,1.25
2026-07-19,64.35,102.96,50.96,63.70,1.6,1.25
2026-07-26,66.78,106.85,69.46,86.82,1.6,1.25
2026-08-02,62.07,99.31,64.41,80.51,1.6,1.25


### Reading the Mismatch
Both pairs agree in 596 of the 626 weeks: the values are literally the same number in both files. The 30 mismatched weeks are all the weeks from 6/7/2026 to the end of the data, and both files disagree in exactly the same weeks.

Keep `fuel_cost_index` and `commodity_price_index` from `weekly_route_operations.csv` and add only the 4 new prices (gas, steel, wheat, copper), so the same information isn't stored twice under different names.

In [37]:
merged_df = merged_df.merge(
    commodity_market_df[["date", "natural_gas_price", "steel_price", "wheat_price", "copper_price"]],
    on="date", how="left", validate="many_to_one",
)

print("Shape:", merged_df.shape)

Shape: (31300, 39)


## 10. Summarize Geopolitical Events
Events are individual dated rows tagged with an `affected_region`, so they can't be joined directly. They need to be:
1. mapped to the same regions as the route countries
2. rolled up into weekly features
3. joined to each route by its origin and destination region

### Map Country Regions to Event Regions
Update current groupings into broader groupings (collapse upward)

In [38]:
region_to_event_region = {
    "East Asia & Pacific": "Asia",
    "South Asia": "Asia",
    "Europe & Central Asia": "Europe",
    "Latin America & Caribbean": "South America",
    "North America": "North America",
}

merged_df["origin_event_region"] = merged_df["origin_region"].map(region_to_event_region)
merged_df["destination_event_region"] = merged_df["destination_region"].map(region_to_event_region)

print("Unmapped regions:", merged_df[["origin_event_region", "destination_event_region"]].isna().sum().sum())
geopolitical_events_df["affected_region"].value_counts()

Unmapped regions: 0


affected_region
Europe           1730
Middle East      1693
Africa           1653
South America    1649
Asia             1640
North America    1637
Global              1
Name: count, dtype: int64

### Can Event Duration Be Predicted?
We need to test whether unrest near a route helps explain `route_status == "Disrupted"`. `geopolitical_events.csv` is the only file holding that information.

But the main issue is that it arrives as 10,000 individual events, which can't be joined to a weekly table until they're summarized into per-week numbers.

Each event has a start `date` and a `duration_days`. In real life, you don't know how long an event will last on the day it starts. But we can try  observing if similar events lasted similar lengths of time, we could potentially make estimate durations from past events of the same `event_type` or `severity`.

**Note on the 3 major shocks:** besides the 10,000 routine events, the file ends with 3 extra rows for real-world crises:
- `MAJOR001`: COVID Supply Shock (2020-03-15, Global, 500 days)
- `MAJOR002`: Russia Ukraine War (2022-02-24, Europe, 1,000 days)
- `MAJOR003`: Red Sea Crisis (2024-01-01, Middle East, 400 days)

They are excluded from this check because they don't look like the routine events:
- **Much longer:** routine events last 7 to 364 days, while these last 400 to 1,000 days, so they would pull up the averages.
- **One of a kind:** each has its own `event_type` that appears only once, so there are no similar past events to estimate their duration from.

Instead, they are known events with fixed dates, so they are handled separately in *Major Shock Flags* below.

In [39]:
routine_events_df = geopolitical_events_df[~geopolitical_events_df["event_id"].str.startswith("MAJOR")].copy()

for group in ["event_type", "severity", "affected_region"]:
    group_means = routine_events_df.groupby(group)["duration_days"].mean()
    print(f"Mean duration by {group}: {group_means.min():.0f} to {group_means.max():.0f} days")

# Average error if we guessed each event's duration from the mean of its group
duration_error = lambda guess: (routine_events_df["duration_days"] - guess).abs().mean()
print(f"\nError guessing the overall mean: {duration_error(routine_events_df['duration_days'].mean()):.1f} days")
print(f"Error guessing the event_type mean: {duration_error(routine_events_df.groupby('event_type')['duration_days'].transform('mean')):.1f} days")
print(f"Error guessing the event_type + severity mean: {duration_error(routine_events_df.groupby(['event_type', 'severity'])['duration_days'].transform('mean')):.1f} days")

Mean duration by event_type: 183 to 189 days
Mean duration by severity: 183 to 193 days
Mean duration by affected_region: 183 to 192 days

Error guessing the overall mean: 90.1 days
Error guessing the event_type mean: 90.1 days
Error guessing the event_type + severity mean: 89.5 days


### Active Events per Week
The check above shows that durations can't be predicted. Across all 10,000 routine events, the average `duration_days` is about 186. When the events are grouped by `event_type`, `severity` or `affected_region`, every group still averages between 183 and 193 days. Knowing an event's details doesn't tell you anything more about how long it will last.

Instead, we use each event's observed status: while an event is ongoing, you can see that it's still ongoing, even if you don't know when it will end.

An event is active from `date` to `date + duration_days`. It counts toward a week if it's active on any of that week's 7 days (the 6 days before the week's `date` through the `date` itself).

Africa and Middle East events are dropped. Every route starts and ends in Asia, Europe, North America or South America, so no route can be matched to events in Africa or the Middle East. A ship might pass through those regions along the way, but the data only gives each route's start and end, not its path. Rather than guess, we keep only events in the 4 regions that routes connect.

For each of those regions and each week, we keep track of two metrics:
- `active_event_count`: how many events are active
- `active_event_risk`: the total `risk_increase` of those events

Note: Avoid leaking future information. Each week's values count events that were active at *any point* during that week. That includes events that ended partway through the week. Before a week starts, you can't know which events will end during it. If the model predicted week *t*'s `route_status` from week *t*'s own event features, it would use information that wasn't available in advance. To prevent this, only use week *t*'s event features to predict week *t+1*'s `route_status`. By the time week *t+1* starts, everything about week *t* is already known.

In [40]:
# Keep only events in a region some route starts or ends in (drops Africa and Middle East)
route_event_regions = set(region_to_event_region.values())
route_events_df = routine_events_df[routine_events_df["affected_region"].isin(route_event_regions)].copy()
print("Events kept:", len(route_events_df), "of", len(routine_events_df))

route_events_df["end_date"] = route_events_df["date"] + pd.to_timedelta(route_events_df["duration_days"], unit="D")
week_dates = pd.DatetimeIndex(weekly_route_operations_df["date"].unique()).sort_values().to_numpy()

# One row per week, one column per event: True if the event overlaps the 7 days ending on that week's date
is_active = (
    (route_events_df["date"].to_numpy() <= week_dates[:, None])
    & (route_events_df["end_date"].to_numpy() >= week_dates[:, None] - np.timedelta64(6, "D"))
)

# One row per event, one column per region: 1 in the column of the event's region
event_regions = pd.get_dummies(route_events_df["affected_region"], dtype=int)
event_region_risk = event_regions.mul(route_events_df["risk_increase"], axis=0)

active_count = pd.DataFrame(is_active @ event_regions.to_numpy(), index=week_dates, columns=event_regions.columns)
active_risk = pd.DataFrame(is_active @ event_region_risk.to_numpy(), index=week_dates, columns=event_regions.columns).round(2)

region_event_features_df = pd.concat(
    [active_count.stack().rename("active_event_count"), active_risk.stack().rename("active_event_risk")], axis=1
).rename_axis(["date", "event_region"]).reset_index()

active_count.describe().round(1)

Events kept: 6655 of 10000


,Asia,Europe,North America,South America
count,626.0,626.0,626.0,626.0
mean,69.5,74.1,68.9,70.3
std,12.4,13.1,12.9,11.5
min,3.0,0.0,3.0,1.0
25%,66.0,70.0,63.0,66.0
50%,70.0,75.0,70.0,72.0
75%,77.0,81.0,77.0,77.0
max,89.0,96.0,88.0,94.0


### Join Event Features to Routes
Each route gets 2 copies of the 2 features: one for its origin region (`origin_`) and one for its destination region (`destination_`).

**Warm-up period:** The event file starts on 2015-01-01, so it has no events that started in 2014. Some of those 2014 events would still have been active in early 2015.

`events_warmup` flags this incomplete period:
- `1`: the week is before 2015-12-31 (the first 52 weeks), so its event counts may be too low
- `0`: the week is on or after 2015-12-31, so its event counts are complete

The modeling step can decide whether to drop the flagged weeks.

In [41]:
event_feature_columns = ["active_event_count", "active_event_risk"]

for side in ["origin", "destination"]:
    side_features_df = region_event_features_df.rename(columns={"event_region": f"{side}_event_region"})
    side_features_df = side_features_df.rename(columns={col: f"{side}_{col}" for col in event_feature_columns})
    merged_df = merged_df.merge(side_features_df, on=["date", f"{side}_event_region"], how="left", validate="many_to_one")

warmup_end = geopolitical_events_df["date"].min() + pd.Timedelta(days=route_events_df["duration_days"].max())
merged_df["events_warmup"] = (merged_df["date"] < warmup_end).astype(int)

print("Shape:", merged_df.shape)
print("Warm-up weeks:", merged_df.loc[merged_df["events_warmup"] == 1, "date"].nunique(), "| ends before", warmup_end.date())

Shape: (31300, 46)
Warm-up weeks: 52 | ends before 2015-12-31


### Major Shock Flags
Add a 0/1 flag for each major shock, using the same active rule as routine events: 1 from the start date until `duration_days` later. They get their own flags instead of being counted with routine events because each one is a single, much larger event.

Shocks are matched to routes the same way as routine events:
- **COVID Supply Shock** (Global): flagged on every route.
- **Russia Ukraine War** (Europe): flagged only on routes that start or end in Europe.
- **Red Sea Crisis** (Middle East): dropped, for the same reason as the routine Africa and Middle East events. No route starts or ends in the Middle East, and the data doesn't say which routes pass through it.

In [42]:
major_events_df = geopolitical_events_df[geopolitical_events_df["event_id"].str.startswith("MAJOR")]

# Same rule as routine events: drop shocks in regions no route starts or ends in (removes the Red Sea Crisis)
major_events_df = major_events_df[major_events_df["affected_region"].isin(route_event_regions | {"Global"})]

for _, event in major_events_df.iterrows():
    start = event["date"]
    end = start + pd.Timedelta(days=event["duration_days"])
    flag_name = event["event_type"].lower().replace(" ", "_") + "_active"

    # Global shocks apply to every route; regional shocks only to routes starting or ending in that region
    if event["affected_region"] == "Global":
        route_in_region = True
    else:
        route_in_region = (merged_df["origin_event_region"] == event["affected_region"]) | (merged_df["destination_event_region"] == event["affected_region"])

    merged_df[flag_name] = (merged_df["date"].between(start, end) & route_in_region).astype(int)
    flagged_routes = merged_df.loc[merged_df[flag_name] == 1, "route_id"].nunique()
    print(f"{flag_name}: {start.date()} to {end.date()}, {merged_df.drop_duplicates('date')['date'].between(start, end).sum()} weeks, {flagged_routes} routes")

major_events_df

covid_supply_shock_active: 2020-03-15 to 2021-07-28, 72 weeks, 50 routes
russia_ukraine_war_active: 2022-02-24 to 2024-11-20, 143 weeks, 26 routes


,event_id,date,event_type,severity,affected_region,duration_days,risk_increase
10000,MAJOR001,2020-03-15,COVID Supply Shock,10,Global,500,50.0
10001,MAJOR002,2022-02-24,Russia Ukraine War,10,Europe,1000,60.0


## 11. Sanity Checks
The merged table should still have one row per route-week (31,300) with no new missing values.

In [43]:
print("Rows:", len(merged_df), "| matches weekly_route_operations_df:", len(merged_df) == len(weekly_route_operations_df))
print("Duplicate route-weeks:", merged_df.duplicated(["route_id", "date"]).sum())
print("Missing values:", merged_df.isna().sum().sum())
print("Columns:", merged_df.shape[1])

Rows: 31300 | matches weekly_route_operations_df: True
Duplicate route-weeks: 0
Missing values: 0
Columns: 48


### Disrupted Rate by Year
Disrupted stays around 2–3% until 2026, then jumps. Almost all of that jump is after 6/7/2026, the same weeks where the commodity values change scale (see section 9). This matters for the train/test split because the final test period will look very different from training.

In [44]:
is_disrupted = (merged_df["route_status"] == "Disrupted").astype(int)

is_disrupted.groupby(merged_df["date"].dt.year).mean().mul(100).round(1).rename("disrupted_percent")

date
2015     2.8
2016     2.5
2017     2.3
2018     2.7
2019     2.6
2020     2.5
2021     2.4
2022     2.3
2023     2.3
2024     1.8
2025     2.0
2026    11.9
Name: disrupted_percent, dtype: float64

## 12. Order Columns & Save a New CSV
The merge added columns in the order the files were joined, which is hard to read. Before saving, the columns are grouped by topic, with the most useful groups first:

| # | Group | Columns | What it tells you |
|---|---|---|---|
| 1 | **Key & target** | `date`, `route_id`, `route_status` | Which route and week the row is, and the status the model predicts |
| 2 | **Route** | origin/destination country, shipping method, trade type, distance, transit days | What the route is. These stay the same every week |
| 3 | **Weekly operations** | delay, congestion, containers, weather, geopolitical risk, volume, cost, emissions | How the route performed that week |
| 4 | **Geopolitical events** | origin/destination event counts and risk, major shock flags, `events_warmup` | Unrest near the route that week (built in section 10) |
| 5 | **Market prices** | fuel, commodity index, natural gas, steel, wheat, copper | Global prices that week. Same for every route |
| 6 | **Origin country** | region, income group, population, GDP, trade and logistics scores | Background on the origin country. Never changes |
| 7 | **Destination country** | same as origin | Background on the destination country. Never changes |

Inside each group, the columns most likely to relate to disruptions come first.

The code checks that every column is placed exactly once. If a new column is added earlier in the notebook and not listed here, it raises an error instead of silently dropping the column.

In [45]:
column_groups = {
    "key_and_target": ["date", "route_id", "route_status"],
    "route": [
        "origin_country", "destination_country", "shipping_method", "trade_route_type",
        "distance_km", "estimated_transit_days",
    ],
    "weekly_operations": [
        "shipping_delay_days", "port_congestion_index", "container_availability_index",
        "weather_disruption_score", "geopolitical_risk_score",
        "trade_volume_tonnes", "freight_cost_usd", "carbon_emissions_tonnes",
    ],
    "geopolitical_events": [
        "origin_active_event_count", "origin_active_event_risk",
        "destination_active_event_count", "destination_active_event_risk",
        "covid_supply_shock_active", "russia_ukraine_war_active", "events_warmup",
    ],
    "market_prices": [
        "fuel_cost_index", "commodity_price_index",
        "natural_gas_price", "steel_price", "wheat_price", "copper_price",
    ],
}

# Origin and destination country details share the same layout
for side in ["origin", "destination"]:
    column_groups[f"{side}_country"] = [
        f"{side}_region", f"{side}_event_region", f"{side}_income_group",
        f"{side}_port_capacity_index", f"{side}_logistics_performance_index", f"{side}_trade_dependency_score",
        f"{side}_gdp_per_capita", f"{side}_population", f"{side}_iso3",
    ]

column_order = [column for group in column_groups.values() for column in group]

# Every column must be placed exactly once
missing_columns = set(merged_df.columns) - set(column_order)
unknown_columns = set(column_order) - set(merged_df.columns)
assert len(column_order) == len(set(column_order)), "A column is listed twice"
assert not missing_columns, f"Columns not placed in a group: {missing_columns}"
assert not unknown_columns, f"Listed columns that don't exist: {unknown_columns}"

merged_df = merged_df[column_order]

for group, columns in column_groups.items():
    print(f"{group} ({len(columns)}): {', '.join(columns)}")

key_and_target (3): date, route_id, route_status
route (6): origin_country, destination_country, shipping_method, trade_route_type, distance_km, estimated_transit_days
weekly_operations (8): shipping_delay_days, port_congestion_index, container_availability_index, weather_disruption_score, geopolitical_risk_score, trade_volume_tonnes, freight_cost_usd, carbon_emissions_tonnes
geopolitical_events (7): origin_active_event_count, origin_active_event_risk, destination_active_event_count, destination_active_event_risk, covid_supply_shock_active, russia_ukraine_war_active, events_warmup
market_prices (6): fuel_cost_index, commodity_price_index, natural_gas_price, steel_price, wheat_price, copper_price
origin_country (9): origin_region, origin_event_region, origin_income_group, origin_port_capacity_index, origin_logistics_performance_index, origin_trade_dependency_score, origin_gdp_per_capita, origin_population, origin_iso3
destination_country (9): destination_region, destination_event_region

### Save the Merged Data
Save to `data/preprocessed/supply_chain_disruption.csv` so feature engineering and modeling can start from here.

Two changes are made on the way out, for readability only:
- `date` is written as `MM/DD/YYYY`
- `route_id` drops its zero padding: `R00001` -> `R1`

In [46]:
export_df = merged_df.copy()
export_df["route_id"] = "R" + export_df["route_id"].str.lstrip("R").astype(int).astype(str)

os.makedirs("../data/preprocessed", exist_ok=True)
export_df.to_csv("../data/preprocessed/supply_chain_disruption.csv", index=False, date_format="%m/%d/%Y")

print("Saved", export_df.shape, "to ../data/preprocessed/supply_chain_disruption.csv")
print("route_id sample:", export_df["route_id"].unique()[:3].tolist(), "...", export_df["route_id"].unique()[-1])

Saved (31300, 48) to ../data/preprocessed/supply_chain_disruption.csv
route_id sample: ['R1', 'R2', 'R3'] ... R50
